# Forecasting  
**Unidad 2 – Modelos Clásicos y de Aprendizaje Profundo para Predicción**  
Este notebook aplica técnicas como ARIMA, Prophet


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA

from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Simulated data: Monthly electricity demand in Peru (dummy example)
date_rng = pd.date_range(start='1/1/2010', end='12/1/2022', freq='MS')
np.random.seed(42)
demand = np.random.poisson(lam=200, size=len(date_rng)) + np.sin(np.arange(len(date_rng)) / 6.0) * 15
df = pd.DataFrame({'date': date_rng, 'demand': demand})
df.set_index('date', inplace=True)

# Plot
df.plot(title='Peruvian Electricity Demand (simulated)', figsize=(10,4))
plt.ylabel('Demand')
plt.grid(True)
plt.show()


In [ ]:
result = seasonal_decompose(df['demand'], model='additive')
result.plot()
plt.suptitle("Decomposición Aditiva de la Serie", fontsize=14)
plt.show()


In [ ]:
model = ARIMA(df['demand'], order=(1,1,1))
model_fit = model.fit()
forecast = model_fit.forecast(steps=12)

plt.figure(figsize=(10,4))
plt.plot(df.index, df['demand'], label='Original')
plt.plot(pd.date_range(df.index[-1], periods=12, freq='MS'), forecast, label='ARIMA Forecast')
plt.legend()
plt.title("ARIMA Forecast")
plt.show()


In [ ]:
df_prophet = df.reset_index().rename(columns={'date': 'ds', 'demand': 'y'})
m = Prophet()
m.fit(df_prophet)
future = m.make_future_dataframe(periods=12, freq='MS')
forecast = m.predict(future)
fig = m.plot(forecast)
